<a href="https://colab.research.google.com/github/2303a51060Nirnaya/High_performance_computing-Hcp-/blob/main/lab_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

SUM OF 0 - 100 NUMBERS


In [9]:
%%writefile sum.c
#include <stdlib.h>
#include <mpi.h>
#include <stdio.h>
#define START 0
#define END 100

int main(int argc, char** argv){
    int rank, size;

    /* initializing MPI */
    MPI_Init(&argc, &argv);

    /* get process id and number of processors */
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    int start = ((END - START) / size) * rank;
    int end = start + ((END - START) / size) - 1;

    if (rank == size - 1){
        end = END;
    }

    int sum = 0, i;

    for(i = start; i <= end; i++){
        sum += i;
    }
    printf("process %d: sum(%d,%d) = %d\n", rank, start, end, sum);
    if (rank == 0){
        int partial;
        MPI_Status status;

        for(i = 1; i < size; i++){
            MPI_Recv(&partial, 1, MPI_INT, i, 0, MPI_COMM_WORLD, &status);
            sum += partial;
        }
    }
    else{
        MPI_Send(&sum, 1, MPI_INT, 0, 0, MPI_COMM_WORLD);
    }

    if(rank == 0){
        printf("The final sum = %d\n", sum);
    }

    MPI_Finalize();
    return 0;
}

!mpicc sum.c -o sum
!mpirun -np 4 ./sum

Writing sum.c
